## Ejercicio 2: Escalamiento de tickets de soporte técnico

### Ficha PEAS

| Componente PEAS | Descripción |
| :--- | :--- |
| **Percepción (S)** | Tupla de datos: `tiempo_espera_minutos` (entero), `nivel_urgencia` ("baja", "media", "alta") y estado `cliente_premium` (Booleano). |
| **Acciones (A)** | Asignar ruta: "escalar a nivel 1", "escalar a nivel 2" o "escalar a nivel 3", con su motivo. |
| **Entorno (E)** | Sistema gestor de incidencias (Helpdesk) en tiempo real. |
| **Objetivo** | Resolver tickets optimizando tiempos de respuesta según SLA y prioridad de clientes. |
| **Medida de desempeño (P)** | Tiempo promedio de resolución (MTTR), porcentaje de SLA cumplido y nivel CSAT. |

### Justificación de las reglas


Se redirige de inmediato al Nivel 3 cualquier caso con urgencia alta o a clientes premium con demoras superiores a 30 minutos para evitar penalizaciones en acuerdos SLA críticos. El Nivel 2 absorbe los casos de urgencia media, solicitudes regulares de clientes premium o cualquier ticket con espera prolongada superior a 45 minutos. Finalmente, las solicitudes con urgencia baja de usuarios estándar con poco tiempo de espera son enviadas al Nivel 1 para su resolución rápida en mesa de entrada.

### Código

In [6]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    """
    Determina el nivel de escalamiento de un ticket de soporte técnico.
    Retorna una tupla: (accion, motivo).
    """
    # Convertimos el texto a minúsculas para evitar fallos con mayúsculas ("ALTA", "Alta", etc.)
    urgencia = nivel_urgencia.lower()

    # Prioridad Máxima (Nivel 3): Urgencia alta O (cliente premium esperando más de 30 min)
    if urgencia == "alta" or (cliente_premium and tiempo_espera_minutos > 30):
        # Determinamos cuál de las dos razones activó el Nivel 3
        if urgencia == "alta":
            motivo = "urgencia alta reportada por el cliente"
        else:
            motivo = f"cliente premium con tiempo de espera critico ({tiempo_espera_minutos} min)"
        
        return "escalar a nivel 3", motivo

    # Prioridad Intermedia (Nivel 2): Urgencia media, cliente premium o espera mayor a 45 min
    elif urgencia == "media" or cliente_premium or tiempo_espera_minutos > 45:
        # Evaluamos en orden para asignar el motivo correspondiente
        if urgencia == "media":
            motivo = "urgencia media requiere soporte especializado intermedio"
        elif cliente_premium:
            motivo = "priorizacion asignada por estado de cliente premium"
        else:
            motivo = f"tiempo de espera excedido para flujo estandar ({tiempo_espera_minutos} min)"
        
        return "escalar a nivel 2", motivo

    # Si no cumple ninguna condición de prioridad superior, asigna Nivel 1
    else:
        return "escalar a nivel 1", f"atencion regular de nivel 1 (urgencia baja, espera: {tiempo_espera_minutos} min)"

### Simulación y pruebas

In [5]:
# Lista de tuplas con diferentes escenarios de prueba:
# Formato de cada tupla: (tiempo_espera_minutos, nivel_urgencia, cliente_premium)
casos_prueba_soporte = [
    (10, "alta", False),   # Escalar a Nivel 3 (Urgencia alta directa)
    (35, "baja", True),    # Escalar a Nivel 3 (Urgencia baja pero cliente premium con espera > 30 min)
    (15, "media", False),  # Escalar a Nivel 2 (Urgencia media)
    (10, "baja", True),    # Escalar a Nivel 2 (Cliente premium con espera corta)
    (50, "baja", False),   # Escalar a Nivel 2 (Urgencia baja pero tiempo > 45 min)
    (20, "baja", False)    # Escalar a Nivel 1 (Atención estándar básica)
]

print("=== PRUEBAS DEL AGENTE DE SOPORTE TÉCNICO ===")

# Recorremos la lista de pruebas:
# 'i' guarda el número de caso (empezando en 1 gracias al argumento de enumerate)
# '(espera, urgencia, premium)' desempaqueta los tres datos de la tupla actual
for i, (espera, urgencia, premium) in enumerate(casos_prueba_soporte, 1):
    
    # Enviamos los datos a la función y recibimos la decisión y la razón del escalamiento
    accion, motivo = agente_soporte(espera, urgencia, premium)
    
    # Imprimimos los datos de entrada, la decisión en mayúsculas (.upper()) y el motivo explicativo
    print(f"Caso {i}: Espera={espera}m, Urgencia={urgencia}, Premium={premium} | Decision: {accion.upper()} | Motivo: {motivo}")

=== PRUEBAS DEL AGENTE DE SOPORTE TÉCNICO ===
Caso 1: Espera=10m, Urgencia=alta, Premium=False | Decision: ESCALAR A NIVEL 3 | Motivo: urgencia alta reportada por el cliente
Caso 2: Espera=35m, Urgencia=baja, Premium=True | Decision: ESCALAR A NIVEL 3 | Motivo: cliente premium con tiempo de espera critico (35 min)
Caso 3: Espera=15m, Urgencia=media, Premium=False | Decision: ESCALAR A NIVEL 2 | Motivo: urgencia media requiere soporte especializado intermedio
Caso 4: Espera=10m, Urgencia=baja, Premium=True | Decision: ESCALAR A NIVEL 2 | Motivo: priorizacion asignada por estado de cliente premium
Caso 5: Espera=50m, Urgencia=baja, Premium=False | Decision: ESCALAR A NIVEL 2 | Motivo: tiempo de espera excedido para flujo estandar (50 min)
Caso 6: Espera=20m, Urgencia=baja, Premium=False | Decision: ESCALAR A NIVEL 1 | Motivo: atencion regular de nivel 1 (urgencia baja, espera: 20 min)
